In [4]:
# ========================================
# CELL 1: Gold Layer Configuration
# ========================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("✅ Gold Layer Pipeline Started")
print(f"📅 {datetime.now()}")

StatementMeta(, 4ee465b3-9c46-454a-8299-c7981bc5eac3, 6, Finished, Available, Finished, False)

✅ Gold Layer Pipeline Started
📅 2026-05-12 22:55:16.283839


In [ ]:
# ========================================
# CELL 2: Load Silver Layer Tables
# ========================================

print("📥 Loading Silver layer tables...\n")

# Load tables
sales_df = spark.table("silver_sales")
products_df = spark.table("silver_products")
customers_df = spark.table("silver_customers")

print(f"✅ Sales: {sales_df.count():,} records")
print(f"✅ Products: {products_df.count():,} products")
print(f"✅ Customers: {customers_df.count():,} customers")

print("\n🔍 Sales Sample:")
sales_df.select("order_date", "customer_name", "product_name", "sales", "profit").show(5)

In [ ]:
# ========================================
# CELL 3: Gold Table - Daily Sales Summary
# ========================================

print("🔄 Creating Daily Sales Summary...\n")

gold_daily_sales = (
    sales_df
    .groupBy(
        "order_date",
        "order_year",
        "order_month",
        "region",
        "category"
    )
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("profit").alias("total_profit"),
        F.sum("quantity").alias("total_quantity"),
        F.count("order_id").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("profit_margin").alias("avg_profit_margin"),
        F.sum("discount_amount").alias("total_discount"),
        F.max("sales").alias("max_sale"),
        F.min("sales").alias("min_sale")
    )
    
    # Derived Metrics
    .withColumn("avg_order_value", 
                F.round(F.col("total_sales") / F.col("order_count"), 2))
    .withColumn("revenue_per_customer", 
                F.round(F.col("total_sales") / F.col("unique_customers"), 2))
    .withColumn("avg_items_per_order",
                F.round(F.col("total_quantity") / F.col("order_count"), 2))
    .withColumn("created_timestamp", F.current_timestamp())
)

# Write to Gold Layer
print("💾 Writing to Delta...")
(gold_daily_sales.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("order_year", "order_month")
 .save("Files/gold/daily_sales_summary")) # ✅ Files/ prefix

# Create managed table pointing to Files location - silver sales table 
gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_sales_summary")

print(f"✅ gold_daily_sales_summary created: {gold_daily_sales.count():,} records")
print("\n🔍 Sample:")
gold_daily_sales.orderBy(F.desc("total_sales")).show(5)

In [ ]:
# ========================================
# CELL 4: Gold Table - Customer Lifetime Value (CLV)
# ========================================

print("🔄 Creating Customer Lifetime Value analysis...\n")

gold_customer_clv = (
    sales_df
    .groupBy("customer_id", "customer_name", "segment", "region", "state")
    .agg(
        # Financial Metrics
        F.sum("sales").alias("lifetime_sales"),
        F.sum("profit").alias("lifetime_profit"),
        F.sum("quantity").alias("lifetime_quantity"),
        F.count("order_id").alias("total_orders"),
        
        # Time Metrics
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
        
        # Averages
        F.avg("sales").alias("avg_order_value"),
        F.avg("profit_margin").alias("avg_profit_margin"),
        F.avg("discount").alias("avg_discount_rate")
    )
    
    # RFM Calculations
    .withColumn("days_as_customer", 
                F.datediff(F.col("last_order_date"), F.col("first_order_date")))
    
    .withColumn("recency_days",
                F.datediff(F.current_date(), F.col("last_order_date")))
    
    # Frequency Score (1-5)
    .withColumn("frequency_score",
                F.when(F.col("total_orders") >= 20, 5)
                .when(F.col("total_orders") >= 15, 4)
                .when(F.col("total_orders") >= 10, 3)
                .when(F.col("total_orders") >= 5, 2)
                .otherwise(1))
    
    # Monetary Score (1-5)
    .withColumn("monetary_score",
                F.when(F.col("lifetime_sales") >= 10000, 5)
                .when(F.col("lifetime_sales") >= 5000, 4)
                .when(F.col("lifetime_sales") >= 2000, 3)
                .when(F.col("lifetime_sales") >= 500, 2)
                .otherwise(1))
    
    # Recency Score (1-5) - lower days = higher score
    .withColumn("recency_score",
                F.when(F.col("recency_days") <= 30, 5)
                .when(F.col("recency_days") <= 90, 4)
                .when(F.col("recency_days") <= 180, 3)
                .when(F.col("recency_days") <= 365, 2)
                .otherwise(1))
    
    # Overall RFM Score
    .withColumn("rfm_score",
                F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score"))
    
    # Customer Segmentation
    .withColumn("customer_value_segment",
                F.when((F.col("recency_score") >= 4) & 
                      (F.col("frequency_score") >= 4) & 
                      (F.col("monetary_score") >= 4), "Champions")
                .when((F.col("frequency_score") >= 3) & 
                      (F.col("monetary_score") >= 3), "Loyal Customers")
                .when(F.col("recency_score") >= 4, "Promising")
                .when(F.col("recency_days") <= 90, "Active")
                .when(F.col("recency_days") <= 180, "At Risk")
                .otherwise("Churned"))
    
    .withColumn("created_timestamp", F.current_timestamp())
)

# Write to Gold
print("💾 Writing to Delta...")
(gold_customer_clv.write
 .format("delta")
 .mode("overwrite")
 .save("Files/gold/customer_lifetime_value"))

# Create managed table pointing to Files location - silver sales table 
gold_customer_clv.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_clv")

print(f"✅ gold_customer_clv created: {gold_customer_clv.count():,} customers")
print("\n🏆 Top 10 Customers by Lifetime Value:")
gold_customer_clv.orderBy(F.desc("lifetime_sales")).select(
    "customer_name", "customer_value_segment", "lifetime_sales", 
    "total_orders", "rfm_score"
).show(10, truncate=False)

# Customer Segment Distribution
print("\n📊 Customer Segment Distribution:")
gold_customer_clv.groupBy("customer_value_segment").agg(
    F.count("*").alias("customer_count"),
    F.sum("lifetime_sales").alias("total_sales")
).orderBy(F.desc("total_sales")).show()

In [ ]:
# ========================================
# CELL 5: Gold Table - Product Performance
# ========================================

print("🔄 Creating Product Performance analysis...\n")

gold_product_performance = (
    sales_df
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "subcategory"
    )
    .agg(
        F.sum("sales").alias("total_revenue"),
        F.sum("profit").alias("total_profit"),
        F.sum("quantity").alias("units_sold"),
        F.count("order_id").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("profit_margin").alias("avg_profit_margin"),
        F.avg("discount").alias("avg_discount_rate"),
        F.max("sales").alias("max_single_sale"),
        F.min("sales").alias("min_single_sale")
    )
    
    # Calculated Metrics
    .withColumn("revenue_per_unit", 
                F.round(F.col("total_revenue") / F.col("units_sold"), 2))
    .withColumn("profit_per_unit",
                F.round(F.col("total_profit") / F.col("units_sold"), 2))
    
    # Rankings
    .withColumn("revenue_rank", 
                F.row_number().over(Window.orderBy(F.desc("total_revenue"))))
    .withColumn("profit_rank",
                F.row_number().over(Window.orderBy(F.desc("total_profit"))))
    .withColumn("volume_rank",
                F.row_number().over(Window.orderBy(F.desc("units_sold"))))
    
    # Performance Categories
    .withColumn("performance_category",
                F.when(F.col("revenue_rank") <= 20, "Top Seller")
                .when(F.col("revenue_rank") <= 100, "Good Performer")
                .when(F.col("revenue_rank") <= 300, "Average")
                .otherwise("Low Performer"))
    
    .withColumn("profitability_category",
                F.when(F.col("avg_profit_margin") >= 30, "High Margin")
                .when(F.col("avg_profit_margin") >= 15, "Medium Margin")
                .when(F.col("avg_profit_margin") >= 0, "Low Margin")
                .otherwise("Loss Making"))
    
    .withColumn("created_timestamp", F.current_timestamp())
)

# Write to Gold
print("💾 Writing to Delta...")
(gold_product_performance.write
 .format("delta")
 .mode("overwrite")
 .save("Files/gold/product_performance"))



# Create managed table pointing to Files location - gold product performance table 
gold_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_product_performance")

print(f"✅ gold_product_performance created: {gold_product_performance.count():,} products")
print("\n🏆 Top 10 Products by Revenue:")
gold_product_performance.orderBy("revenue_rank").select(
    "revenue_rank", "product_name", "category", "total_revenue", 
    "units_sold", "performance_category"
).show(10, truncate=False)

# Category Performance
print("\n📊 Category Performance:")
gold_product_performance.groupBy("category").agg(
    F.sum("total_revenue").alias("category_revenue"),
    F.sum("units_sold").alias("category_units"),
    F.avg("avg_profit_margin").alias("category_margin")
).orderBy(F.desc("category_revenue")).show()

In [ ]:
# ========================================
# CELL 6: Gold Table - Regional Performance
# ========================================

print("🔄 Creating Regional Performance analysis...\n")

gold_regional_performance = (
    sales_df
    .groupBy("region", "state", "order_year", "order_quarter")
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("profit").alias("total_profit"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.count("order_id").alias("order_count"),
        F.avg("sales").alias("avg_order_value"),
        F.avg("profit_margin").alias("avg_profit_margin")
    )
    
    # Business Metrics
    .withColumn("profit_ratio", 
                F.round((F.col("total_profit") / F.col("total_sales")) * 100, 2))
    .withColumn("sales_per_customer", 
                F.round(F.col("total_sales") / F.col("unique_customers"), 2))
    .withColumn("orders_per_customer",
                F.round(F.col("order_count") / F.col("unique_customers"), 2))
    
    # Regional Rankings
    .withColumn("region_sales_rank",
                F.dense_rank().over(
                    Window.partitionBy("order_year", "order_quarter")
                    .orderBy(F.desc("total_sales"))
                ))
    
    .withColumn("created_timestamp", F.current_timestamp())
)

# Write to Gold
print("💾 Writing to Delta...")
(gold_regional_performance.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("order_year")
 .save("Files/gold/regional_performance"))

# Create managed table pointing to Files location - gold product performance table 
gold_regional_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_regional_performance")

print(f"✅ gold_regional_performance created: {gold_regional_performance.count():,} records")
print("\n🌎 Regional Performance Summary:")
gold_regional_performance.groupBy("region").agg(
    F.sum("total_sales").alias("region_total_sales"),
    F.sum("total_profit").alias("region_total_profit"),
    F.avg("profit_ratio").alias("avg_profit_ratio")
).orderBy(F.desc("region_total_sales")).show()

In [ ]:
# ========================================
# CELL 7: Executive Summary & Key Metrics
# ========================================

print("\n" + "="*70)
print("📊 GOLD LAYER - EXECUTIVE SUMMARY")
print("="*70)

# Overall Business Metrics
total_sales = sales_df.select(F.sum("sales")).collect()[0][0]
total_profit = sales_df.select(F.sum("profit")).collect()[0][0]
total_orders = sales_df.count()
total_customers = sales_df.select(F.countDistinct("customer_id")).collect()[0][0]
total_products = sales_df.select(F.countDistinct("product_id")).collect()[0][0]

print(f"\n💰 FINANCIAL METRICS:")
print(f" Total Revenue: ${total_sales:,.2f}")
print(f" Total Profit: ${total_profit:,.2f}")
print(f" Overall Profit Margin: {(total_profit/total_sales*100):.2f}%")

print(f"\n📦 OPERATIONAL METRICS:")
print(f" Total Orders: {total_orders:,}")
print(f" Average Order Value: ${(total_sales/total_orders):,.2f}")
print(f" Total Customers: {total_customers:,}")
print(f" Total Products Sold: {total_products:,}")

print(f"\n👥 CUSTOMER METRICS:")
print(f" Orders per Customer: {(total_orders/total_customers):.2f}")
print(f" Revenue per Customer: ${(total_sales/total_customers):,.2f}")

print("\n" + "="*70)
print("✅ GOLD LAYER TABLES SUCCESSFULLY CREATED:")
print("="*70)
print(" 1. gold_daily_sales_summary - Daily aggregated metrics")
print(" 2. gold_customer_clv - Customer lifetime value & RFM")
print(" 3. gold_product_performance - Product rankings & analysis")
print(" 4. gold_regional_performance - Geographic performance")
print("\n🎉 GOLD LAYER PROCESSING COMPLETE - Ready for Power BI!")
print("="*70)